In [0]:
%sql
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT customer_code) AS unique_customers,
    SUM(CASE WHEN is_current = true THEN 1 ELSE 0 END) AS current_records,
    SUM(CASE WHEN is_current = false THEN 1 ELSE 0 END) AS historical_records
FROM fmcg.gold.dim_customers;

In [0]:
%sql
CREATE OR REPLACE TABLE fmcg.gold.dim_customers_scd2_practice
USING DELTA
AS
SELECT *
FROM fmcg.gold.dim_customers;

In [0]:
%sql
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT customer_code) AS unique_customers,
    SUM(CASE WHEN is_current THEN 1 ELSE 0 END) AS current_records
FROM fmcg.gold.dim_customers_scd2_practice;

In [0]:
%sql

WITH source_customers AS (
    SELECT
        CAST(customer_id AS STRING) AS customer_code,
        customer,
        market,
        platform,
        channel
    FROM fmcg.gold.sb_dim_customers
),
current_target AS (
    SELECT
        customer_code,
        customer,
        market,
        platform,
        channel
    FROM fmcg.gold.dim_customers_scd2_practice
    WHERE is_current = true
)
SELECT
    COALESCE(s.customer_code, t.customer_code) AS customer_code,
    s.customer AS source_customer,
    t.customer AS target_customer,
    s.channel AS source_channel,
    t.channel AS target_channel,
    CASE
    WHEN s.customer_code IS NULL THEN 'TARGET_ONLY'
    WHEN t.customer_code IS NULL THEN 'NEW'
    WHEN NOT (
        s.customer <=> t.customer
        AND s.market <=> t.market
        AND s.platform <=> t.platform
        AND s.channel <=> t.channel
    ) THEN 'CHANGED'
    ELSE 'UNCHANGED'
END AS change_type
FROM source_customers s
FULL OUTER JOIN current_target t
    ON s.customer_code = t.customer_code
ORDER BY change_type, customer_code;


In [0]:
%sql
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT customer_code) AS unique_customers,
    SUM(CASE WHEN is_current = true THEN 1 ELSE 0 END) AS current_records,
    SUM(CASE WHEN is_current = false THEN 1 ELSE 0 END) AS historical_records
FROM fmcg.gold.dim_customers_scd2_practice;

In [0]:
%sql
SELECT *
FROM fmcg.gold.dim_customers_scd2_practice
WHERE customer_code = '789101';

In [0]:
%sql
SELECT
    CAST(customer_id AS STRING) AS customer_code,
    customer,
    CASE
        WHEN CAST(customer_id AS STRING) = '789101'
            THEN 'USA'
        ELSE market
    END AS market,
    platform,
    channel
FROM fmcg.gold.sb_dim_customers
WHERE CAST(customer_id AS STRING) = '789101';

In [0]:
%sql
WITH simulated_source AS (
    SELECT
        CAST(customer_id AS STRING) AS customer_code,
        customer,
        CASE
            WHEN CAST(customer_id AS STRING) = '789101'
                THEN 'USA'
            ELSE market
        END AS market,
        platform,
        channel
    FROM fmcg.gold.sb_dim_customers
    WHERE CAST(customer_id AS STRING) = '789101'
),
current_target AS (
    SELECT *
    FROM fmcg.gold.dim_customers_scd2_practice
    WHERE customer_code = '789101'
      AND is_current = true
)
SELECT
    s.customer_code,
    t.market AS old_market,
    s.market AS new_market,
    CASE
        WHEN NOT (
            s.customer <=> t.customer
            AND s.market <=> t.market
            AND s.platform <=> t.platform
            AND s.channel <=> t.channel
        )
        THEN 'CHANGED'
        ELSE 'UNCHANGED'
    END AS change_type
FROM simulated_source s
JOIN current_target t
    ON s.customer_code = t.customer_code;

In [0]:
%sql
UPDATE fmcg.gold.dim_customers_scd2_practice
SET
    effective_end_date = current_timestamp(),
    is_current = false
WHERE customer_code = '789101'
  AND is_current = true;

In [0]:
%sql
SELECT
    customer_code,
    market,
    effective_start_date,
    effective_end_date,
    is_current
FROM fmcg.gold.dim_customers_scd2_practice
WHERE customer_code = '789101';

In [0]:
%sql
INSERT INTO fmcg.gold.dim_customers_scd2_practice (
    customer_code,
    customer,
    market,
    platform,
    channel,
    effective_start_date,
    effective_end_date,
    is_current
)
SELECT
    CAST(customer_id AS STRING),
    customer,
    'USA',
    platform,
    channel,
    current_timestamp(),
    CAST(NULL AS TIMESTAMP),
    true
FROM fmcg.gold.sb_dim_customers
WHERE CAST(customer_id AS STRING) = '789101';

In [0]:
%sql
SELECT
    customer_code,
    customer,
    market,
    effective_start_date,
    effective_end_date,
    is_current
FROM fmcg.gold.dim_customers_scd2_practice
WHERE customer_code = '789101'
ORDER BY effective_start_date;

In [0]:
%sql
SELECT
    customer_code,
    COUNT(*) AS total_versions,
    SUM(CASE WHEN is_current = true THEN 1 ELSE 0 END)
        AS current_versions,
    SUM(CASE WHEN is_current = false THEN 1 ELSE 0 END)
        AS historical_versions
FROM fmcg.gold.dim_customers_scd2_practice
WHERE customer_code = '789101'
GROUP BY customer_code;

In [0]:
%sql
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT customer_code) AS unique_customers,
    SUM(CASE WHEN is_current = true THEN 1 ELSE 0 END)
        AS current_records,
    SUM(CASE WHEN is_current = false THEN 1 ELSE 0 END)
        AS historical_records
FROM fmcg.gold.dim_customers_scd2_practice;

In [0]:
%sql
SELECT
    s.customer_id AS customer_code,
    t.market AS target_market,
    s.market AS source_market,
    CASE
        WHEN s.customer <=> t.customer
         AND s.market <=> t.market
         AND s.platform <=> t.platform
         AND s.channel <=> t.channel
        THEN 'UNCHANGED'
        ELSE 'CHANGED'
    END AS change_type
FROM fmcg.gold.sb_dim_customers s
JOIN fmcg.gold.dim_customers_scd2_practice t
    ON CAST(s.customer_id AS STRING) = t.customer_code
   AND t.is_current = true
WHERE CAST(s.customer_id AS STRING) = '789102';

In [0]:
%sql
SELECT
    customer_code,
    SUM(CASE WHEN is_current = true THEN 1 ELSE 0 END)
        AS current_versions
FROM fmcg.gold.dim_customers_scd2_practice
GROUP BY customer_code
HAVING SUM(CASE WHEN is_current = true THEN 1 ELSE 0 END) <> 1;